# Testes do artigo: risco de reidentificação em populações pequenas

Este notebook acompanha o artigo *Quando a anonimização não basta*. Ele está dividido em três partes:

- **Parte I – As redes bayesianas:** monta e desenha a rede geradora e a rede de reidentificação, com as tabelas de probabilidade;
- **Parte II – Perguntas à rede:** nove inferências feitas diretamente na rede (preditivas, diagnósticas, independência, explicação por exclusão, intervenção, explicação mais relevante e aprendizado de estrutura);
- **Parte III – Reprodução completa:** roda todos os experimentos do artigo (E1 a E7) e mostra as tabelas e figuras. Os números saem idênticos aos do artigo, porque a semente aleatória é fixa (2026).

**Antes de começar**

- **No computador:** este arquivo precisa estar na mesma pasta dos scripts (`sim_reidentificacao.py`, `rede_reidentificacao.py` etc.), e o kernel do Jupyter precisa ser um Python **3.10 a 3.14** com os pacotes do `requirements.txt` (o pgmpy ainda não funciona no 3.15). A célula 0 confere isso.
- Rode as células de cima para baixo (**Shift+Enter**). Enquanto uma célula roda, aparece `[*]` à esquerda; quando termina, aparece um número.
- Tempo total: cerca de 12 minutos. As células mais demoradas estão indicadas.

Todos os dados são sintéticos e gerados pelo próprio código: nenhum dado pessoal é coletado, lido ou vinculado a pessoas.

**No Google Colab:** basta rodar as células em ordem (*Ambiente de execução → Executar tudo*). A primeira célula baixa o repositório do GitHub e instala o pgmpy; no computador local, ela não faz nada.

## Preparação (Google Colab)

In [ ]:
import sys, os
if "google.colab" in sys.modules:
    if not os.getcwd().endswith("reidentificacao-populacoes-pequenas"):
        if not os.path.exists("reidentificacao-populacoes-pequenas"):
            !git clone -q https://github.com/barbozapauloricardo/reidentificacao-populacoes-pequenas
        %cd reidentificacao-populacoes-pequenas
    !pip install -q "pgmpy>=1.1"
    print("Repositório baixado e pacotes instalados.")
else:
    print("Execução local: nada a preparar.")

## 0. Conferência do ambiente

In [ ]:
import sys, platform
from pathlib import Path
import numpy as np, pandas as pd, matplotlib, scipy, pgmpy
from IPython.display import display, Image, Markdown

print("Python    ", sys.version.split()[0], "|", platform.system(), platform.release())
print("numpy     ", np.__version__)
print("pandas    ", pd.__version__)
print("matplotlib", matplotlib.__version__)
print("scipy     ", scipy.__version__)
print("pgmpy     ", pgmpy.__version__)
assert Path("sim_reidentificacao.py").exists(), "Coloque este notebook na pasta dos scripts."
assert sys.version_info[:2] < (3, 15), "Use um kernel Python 3.10 a 3.14 (o pgmpy 1.1 não roda no 3.15)."
pd.set_option("display.precision", 3)

import os, subprocess, time
def rodar(script):
    # roda o script num processo separado, igual a rodá-lo no terminal
    t0 = time.perf_counter()
    env = dict(os.environ, PYTHONIOENCODING="utf-8", MPLBACKEND="Agg")
    r = subprocess.run([sys.executable, "-W", "ignore::FutureWarning", script],
                       capture_output=True, text=True, encoding="utf-8", errors="replace", env=env)
    print(r.stdout)
    if r.returncode != 0:
        print(r.stderr)
        raise RuntimeError(f"{script} terminou com erro (veja a mensagem acima).")
    print(f"[{script} concluído em {time.perf_counter() - t0:.0f} s]")

print("\nAmbiente OK.")

## Os dados sintéticos

Os experimentos geram seus próprios dados a partir da rede bayesiana geradora. A célula abaixo grava dois exemplos na pasta `dados_sinteticos/` para inspeção: uma turma de 30 respondentes, com o tamanho da classe de equivalência de cada registro sob os desenhos D0 a D3 (colunas `F_D0` a `F_D3`; 1 = respondente único), e uma população de 5.000 respondentes.

In [ ]:
import subprocess as _sp, sys as _sys, os as _os
_sp.run([_sys.executable, "gerar_dados_exemplo.py"], capture_output=True,
        env=dict(_os.environ, PYTHONIOENCODING="utf-8"))
turma = pd.read_csv("dados_sinteticos/turma_N30.csv")
display(turma)
print("Respondentes únicos nesta turma:",
      {d: f"{(turma[f'F_{d}'] == 1).mean():.0%}" for d in ["D0", "D1", "D2", "D3"]})

# Parte I – As redes bayesianas

O artigo usa duas redes. A **rede geradora** descreve quem são os respondentes e produz as turmas sintéticas. A **rede de reidentificação** descreve como a reidentificação acontece e é usada para fazer perguntas (inferências) sobre o risco. Esta parte monta as duas redes, desenha os grafos e mostra as tabelas de probabilidade condicional (CPTs).

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import networkx as nx
import matplotlib.pyplot as plt
from pgmpy.inference import VariableElimination, CausalInference
import sim_reidentificacao as sim
import rede_reidentificacao as rr

def desenhar(arestas, pos, titulo, destaque=(), nos_extra=(), tamanho=(7, 3.6)):
    g = nx.DiGraph(); g.add_nodes_from(list(pos)); g.add_edges_from(arestas)
    cores = ["#f4c27a" if n in destaque else "#cfe2f3" for n in g.nodes]
    plt.figure(figsize=tamanho)
    nx.draw_networkx(g, pos, node_color=cores, node_size=2600, font_size=9,
                     edgecolors="#333333", arrowsize=18, width=1.4)
    plt.title(titulo); plt.axis("off"); plt.margins(0.15)
    import io
    buf = io.BytesIO(); fig = plt.gcf(); fig.savefig(buf, format="png", dpi=110, bbox_inches="tight"); plt.close(fig)
    display(Image(data=buf.getvalue()))

## 1. Rede geradora

Sete quase-identificadores. A idade depende do vínculo (bolsistas tendem a ser mais jovens) e a linha de pesquisa depende da formação; os demais atributos são independentes. As probabilidades são hipóteses de modelagem (Tabela 1 do artigo).

In [ ]:
pos_g = {"genero": (0, 1), "raca": (1, 1), "cidade": (2, 1),
         "vinculo": (0, 0), "formacao": (1.6, 0), "idade": (0, -1), "linha": (1.6, -1)}
desenhar([("vinculo", "idade"), ("formacao", "linha")], pos_g, "Rede geradora dos respondentes")

for nome, dist in [("Gênero", sim.GENERO), ("Raça/cor", sim.RACA), ("Vínculo", sim.VINCULO),
                   ("Formação", sim.FORMACAO), ("Cidade", sim.CIDADE)]:
    print(f"P({nome}):", {k: round(v, 3) for k, v in dist.items()})
print("P(Linha | Formação):")
display(pd.DataFrame(sim.P_LINHA, index=sim.LINHAS).round(2))

## 2. Rede de reidentificação

- **D**: desenho do questionário (D0 detalhado … D3 mais restritivo) – é o único nó sob controle do pesquisador (em laranja);
- **N**: tamanho da turma (15, 30, 100, 500);
- **S**: composição da turma (S1 heterogênea, S0 base, S2 homogênea, S3 muito homogênea);
- **F**: tamanho da classe de equivalência do registro (1 = único, 2, 3, 4, 5+);
- **K**: adversário (interno, que conhece a turma; externo, sem conhecimento auxiliar);
- **Re**: o registro é reidentificado?

A CPT de F é **calculada** a partir da rede geradora para as 64 combinações de D, N e S (é o que liga as duas redes). ⏱ A montagem leva cerca de 1 minuto.

In [ ]:
rede = rr.montar_rede()
ve = VariableElimination(rede)
print("Nós:", sorted(rede.nodes()))
print("Arestas:", sorted(rede.edges()))
print("Modelo válido:", rede.check_model())

pos_r = {"D": (0, 1), "N": (1, 1), "S": (2, 1), "F": (1, 0), "K": (2.2, 0), "Re": (1.6, -1)}
desenhar(rede.edges(), pos_r, "Rede bayesiana de reidentificação (D = intervenção do pesquisador)", destaque=("D",))

**As tabelas de probabilidade (CPTs).** D, N e S têm priori uniforme (0,25 cada). P(K = interno) = 0,5. A tabela de Re traduz as hipóteses do artigo: o adversário interno acerta com probabilidade 1/F; o externo só consegue vincular registros únicos, com probabilidade 0,10.

In [ ]:
print(rede.get_cpds("K"))
print(rede.get_cpds("Re"))

# A CPT de F tem 64 colunas; mostramos um recorte: turma de 30 pessoas, composição S0 (base)
cpd_f = rede.get_cpds("F")
iN, iS = rr.N_ST.index(30), rr.S_ST.index("S0")
recorte = pd.DataFrame(cpd_f.values[:, :, iN, iS], index=[f"F={f}" for f in rr.F_ST], columns=rr.D_ST)
display(Markdown("**P(F | D, N = 30, S = S0)** – recorte da CPT de F"))
display(recorte.round(3))

# Parte II – Perguntas à rede (inferências)

Cada pergunta abaixo é respondida por uma consulta à rede (eliminação de variáveis, pgmpy). As respostas correspondem aos resultados das seções 4.5 e 4.6 do artigo.

### Pergunta 1 (preditiva) – Numa turma de 30 pessoas, com o questionário mais restritivo (D3), qual a probabilidade de um colega reidentificar um respondente?

In [ ]:
q = ve.query(["Re"], evidence={"D": "D3", "N": "30", "K": "interno"}, show_progress=False)
print(q)
print(f"Resposta: P(Re = sim | D3, N = 30, interno) = {rr.p(q, 'sim'):.1%}")

### Pergunta 2 (preditiva) – Como esse risco muda com o desenho, o tamanho da turma e o tipo de adversário?

In [ ]:
linhas = []
for k in rr.K_ST:
    for d in rr.D_ST:
        for n in rr.N_ST:
            q = ve.query(["Re"], evidence={"D": d, "N": str(n), "K": k}, show_progress=False)
            linhas.append({"adversário": k, "desenho": d, "N": n, "P(Re)": rr.p(q, "sim")})
t = pd.DataFrame(linhas).pivot_table(index=["adversário", "desenho"], columns="N", values="P(Re)")
display((t * 100).round(1))

### Pergunta 3 (decisão) – Para cada tamanho de turma, algum desenho de questionário mantém o risco abaixo de 50% diante de um colega?

In [ ]:
for n in rr.N_ST:
    riscos = {d: rr.p(ve.query(["Re"], evidence={"D": d, "N": str(n), "K": "interno"}, show_progress=False), "sim")
              for d in rr.D_ST}
    seguros = [d for d, r in riscos.items() if r < 0.5]
    print(f"N = {n:>3}: " + ", ".join(f"{d} {r:.0%}" for d, r in riscos.items()) +
          f"  ->  desenhos com risco < 50%: {seguros or 'nenhum'}")

### Pergunta 4 (diagnóstica) – Um registro foi reidentificado numa turma de 30. O que isso nos diz sobre o adversário, o desenho e a composição da turma?

In [ ]:
ev = {"Re": "sim", "N": "30"}
for v in ["K", "D", "S"]:
    q = ve.query([v], evidence=ev, show_progress=False)
    print(f"P({v} | Re = sim, N = 30):", {s: round(rr.p(q, s), 3) for s in q.state_names[v]})

### Pergunta 5 (independência) – O desenho do questionário e o tipo de adversário são independentes? E depois de observar uma reidentificação?

In [ ]:
print("D ⊥ K ?            ", not rede.is_dconnected("D", "K"))
print("D ⊥ K | Re ?       ", not rede.is_dconnected("D", "K", observed=["Re"]))
print("D ⊥ Re | F ?       ", not rede.is_dconnected("D", "Re", observed=["F"]))
print("Manto de Markov de F:", sorted(rede.get_markov_blanket("F")))

### Pergunta 6 (explicação por exclusão) – Se a reidentificação aconteceu com um questionário restritivo, fica mais provável que o adversário seja um colega?

Como F é um colisor (recebe D, N e S) e Re depende de F e K, observar Re torna D e K dependentes: quanto menos o desenho “explica” a reidentificação, mais ela precisa ser atribuída a quem conhece a turma.

In [ ]:
for d in rr.D_ST:
    q = ve.query(["K"], evidence={"Re": "sim", "N": "30", "D": d}, show_progress=False)
    print(f"P(K = interno | Re = sim, N = 30, D = {d}) = {rr.p(q, 'interno'):.3f}")

### Pergunta 7 (intervenção) – Qual seria o efeito de **mudar** o desenho do questionário, e não apenas de observar programas que já usam desenhos diferentes?

Numa variante ilustrativa, turmas pequenas tendem a usar questionários mais enxutos (aresta N → D). Comparamos P(Re | D), que mistura o efeito do desenho com o do tamanho da turma, com P(Re | do(D)), obtido pelo ajuste da porta dos fundos sobre N.

In [ ]:
rede_c = rr.montar_rede(confundida=True)
ve_c, ci = VariableElimination(rede_c), CausalInference(rede_c)
desenhar(rede_c.edges(), pos_r, "Variante com confundimento N → D", destaque=("D",))
print("Conjuntos de ajuste (porta dos fundos) para D -> Re:", ci.get_all_backdoor_adjustment_sets("D", "Re"))
linhas = []
for d in rr.D_ST:
    obs = rr.p(ve_c.query(["Re"], evidence={"D": d, "K": "interno"}, show_progress=False), "sim")
    do = sum(0.25 * rr.p(ve_c.query(["Re"], evidence={"D": d, "N": str(n), "K": "interno"}, show_progress=False), "sim")
             for n in rr.N_ST)
    linhas.append({"desenho": d, "P(Re | D)": obs, "P(Re | do(D))": do})
t = pd.DataFrame(linhas).set_index("desenho")
display((t * 100).round(1))
print(f"Diferença D0 − D3: observando {100*(t.iloc[0,0]-t.iloc[3,0]):.1f} p.p.; intervindo {100*(t.iloc[0,1]-t.iloc[3,1]):.1f} p.p.")

### Pergunta 8 (explicação mais relevante, MRE) – Dado que houve reidentificação, qual é a melhor explicação?

Busca exaustiva sobre todas as atribuições parciais de {D, N, S, K}, pelo fator de Bayes generalizado (GBF), com a regra de dominância de Yuan, Lim e Lu (2011). Para comparação, a explicação mais provável (MAP).

In [ ]:
lin = rr.conjunta(rede, ["D", "N", "S", "K"], {})
cand, minimas = rr.mre(lin, ["D", "N", "S", "K"], {"Re": "sim"})
fmt = lambda r: ", ".join(f"{k}={v}" for k, v in r["explicacao"].items())
print("Melhores candidatas (por GBF):")
for r in cand[:5]:
    print(f"  {fmt(r):35s} GBF = {r['GBF']:.2f}   P(x | Re) = {r['posteriori']:.3f}")
print("Explicações mínimas (não dominadas):", [fmt(r) for r in minimas])
lin_e = [(st, pv) for st, pv in lin if st["Re"] == "sim"]
mapa = max(lin_e, key=lambda t: t[1])
print("MAP:", {k: v for k, v in mapa[0].items() if k != "Re"},
      f"com probabilidade a posteriori {mapa[1] / rr.prob(lin, {'Re': 'sim'}):.3f}")

### Pergunta 9 (aprendizado de estrutura) – Se não conhecêssemos a rede geradora, os dados de uma turma permitiriam descobri-la?

Aprendemos a estrutura com Hill Climbing (BIC) a partir de 30 e de 5.000 respondentes simulados e desenhamos as redes aprendidas. A rede verdadeira tem apenas as arestas vínculo–idade e formação–linha.

In [ ]:
from pgmpy.estimators import HillClimbSearch, BIC
import aprendizado_estrutura as ae
sim.rng = np.random.default_rng(2026)
for n in (30, 5000):
    dados = ae.preparar(sim.gerar_turma(n))
    modelo = HillClimbSearch(dados).estimate(scoring_method=BIC(dados), show_progress=False)
    aval = ae.avaliar(ae.esqueleto(modelo.edges()))
    desenhar(modelo.edges(), pos_g, f"Rede aprendida com {n} respondentes – "
             f"{aval['verdadeiros_pos']} de 2 arestas verdadeiras, {aval['falsos_pos']} falsa(s)")

Com 30 respondentes, os dados não bastam para revelar a estrutura que os gerou; com 5.000, ela é recuperada. A direção de uma aresta pode aparecer invertida (por exemplo, idade → vínculo): como a rede geradora não tem colisores, a orientação não é identificável a partir de dados observacionais (as duas redes pertencem à mesma classe de equivalência de Markov); por isso a comparação é feita sobre o esqueleto. A Parte III reproduz a análise completa (20 repetições por tamanho, PC e Hill Climbing).

### Experimente você mesmo

Mude os valores e rode a célula para fazer sua própria pergunta à rede. As opções são: `desenho` = "D0" a "D3"; `tamanho` = "15", "30", "100" ou "500" (entre aspas); `adversario` = "interno" ou "externo".

In [ ]:
desenho, tamanho, adversario = "D2", "100", "interno"
q = ve.query(["Re"], evidence={"D": desenho, "N": tamanho, "K": adversario}, show_progress=False)
print(f"P(Re = sim | D={desenho}, N={tamanho}, K={adversario}) = {rr.p(q, 'sim'):.3f}")

# Parte III – Reprodução completa dos experimentos do artigo

As células abaixo rodam os scripts do repositório, cada um num processo separado (como no terminal), e mostram as tabelas e figuras do artigo. Os números saem idênticos aos publicados, porque a semente é fixa.

## 1. Simulação da unicidade (E1 a E3)

A rede geradora produz turmas sintéticas com sete quase-identificadores. Para cada desenho de questionário (D0 a D3), mede-se a proporção de respondentes únicos por simulação de Monte Carlo e por expressões analíticas.

⏱ **Esta célula leva de 3 a 5 minutos e não mostra nada até terminar.**

In [ ]:
rodar("sim_reidentificacao.py")

**E1. Respondentes únicos por tamanho da turma** (Monte Carlo × expressão analítica)

In [ ]:
e1 = pd.read_csv("resultados/E1_tamanho_turma.csv")
tab = e1.pivot(index="N", columns="desenho", values="prop_unicos_MC") * 100
display(tab.round(1).style.format("{:.1f}%").set_caption("Únicos (%) por N e desenho"))
dif = (e1.prop_unicos_MC - e1.prop_unicos_analitico).abs().max() * 100
print(f"Diferença máxima entre Monte Carlo e expressão analítica: {dif:.2f} ponto percentual")

**E2. Acúmulo de quase-identificadores** (N = 30, desenho D0)

In [ ]:
e2 = pd.read_csv("resultados/E2_numero_qi.csv")
e2["únicos (%)"] = (100 * e2.prop_unicos).round(1)
e2["k < 5 (%)"] = (100 * e2.prop_k_menor_5).round(1)
display(e2[["num_QI", "QI_incluidos", "únicos (%)", "k < 5 (%)"]])

**E3. Resposta incompleta** (60% de adesão, N = 30)

In [ ]:
e3 = pd.read_csv("resultados/E3_resposta_incompleta.csv")
display((e3.set_index("desenho")[["respondentes", "prop_unicos_na_turma",
         "precisao_unico_amostra", "risco_medio_respondentes"]]).round(3))

## 2. Sensibilidade aos parâmetros (E4)

Quatro cenários de composição da turma, do heterogêneo (S1) ao muito homogêneo (S3). ⏱ Cerca de 30 segundos.

In [ ]:
rodar("sensibilidade.py")

In [ ]:
e4 = pd.read_csv("resultados/E4_sensibilidade.csv")
t = e4[e4.N == 30].pivot(index="desenho", columns="cenario", values="prop_unicos") * 100
display(t.round(1).style.format("{:.1f}%").set_caption("Únicos (%) com N = 30"))

## 3. Rede bayesiana de reidentificação (E5 e E6)

Rede: desenho (*D*), tamanho da turma (*N*) e composição (*S*) → classe de equivalência (*F*); *F* e tipo de adversário (*K*) → reidentificação (*Re*). A tabela de *F* é calculada a partir da rede geradora. A célula faz inferência preditiva e diagnóstica, verifica d-separações, compara observar e intervir e busca a explicação mais relevante (MRE).

⏱ **Cerca de 3 a 4 minutos.**

In [ ]:
rodar("rede_reidentificacao.py")

**Tabela 4 do artigo:** P(Re = sim | D, N, K = interno)

In [ ]:
a1 = pd.read_csv("resultados/A1_preditiva.csv")
display((a1.pivot(index="D", columns="N", values="P_Re_sim") * 100).round(1)
        .style.format("{:.1f}%"))

**Inferência diagnóstica:** dado Re = sim numa turma de 30

In [ ]:
a2 = pd.read_csv("resultados/A2_diagnostica.csv")
display(a2.round(3))

**Tabela 5 do artigo:** observar × intervir (variante com confundimento N → D)

In [ ]:
a4 = pd.read_csv("resultados/A4_observar_intervir.csv").set_index("D")
display((a4 * 100).round(1).rename(columns={"P_Re_obs": "P(Re | D)", "P_Re_do": "P(Re | do(D))"}))
print(f"Diferença D0 − D3 ao observar: {100*(a4.P_Re_obs['D0']-a4.P_Re_obs['D3']):.1f} p.p.")
print(f"Diferença D0 − D3 ao intervir: {100*(a4.P_Re_do['D0']-a4.P_Re_do['D3']):.1f} p.p.")

**Explicação mais relevante (MRE):** explicações mínimas dado Re = sim

In [ ]:
display(pd.read_csv("resultados/A5_mre_minimas.csv").round(3))
display(Markdown("Com o adversário interno fixado (busca sobre D, N, S):"))
display(pd.read_csv("resultados/A5_mre_interno_minimas.csv").round(3))

## 4. Sensibilidade da MRE às hipóteses arbitradas

Refaz a MRE variando P(K = interno) e a chance de o adversário externo vincular um registro único. ⏱ Cerca de 1 a 2 minutos.

In [ ]:
rodar("sensibilidade_mre.py")

In [ ]:
a6 = pd.read_csv("resultados/A6_sensibilidade_mre.csv")
display(a6.pivot(index="P_externo_unico", columns="P_K_interno", values="melhor_explicacao"))

## 5. Aprendizado de estrutura (E7)

Com quantos respondentes os algoritmos PC e Hill Climbing (BIC) recuperam a estrutura da rede geradora? ⏱ Cerca de 1 minuto.

In [ ]:
rodar("aprendizado_estrutura.py")

## 6. Figuras do artigo

In [ ]:
rodar("figuras_artigo.py")
for f in ["fig_E1_tamanho", "fig_E2_numero_qi", "fig_E4_sensibilidade", "fig_E7_estrutura"]:
    display(Markdown(f"**{f}**"))
    display(Image(filename=f"figuras/{f}.png", width=480))

## 7. Tempos de execução

In [ ]:
print(open("resultados/ambiente.txt", encoding="utf-8").read())